In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("FuelBurnRegression").getOrCreate()
print(spark.version)

3.4.4


In [2]:
import numpy as np
import pandas as pd

pdf = pd.read_excel("model_df_clean.xlsx")
pdf["Date"] = pd.to_datetime(pdf["Date"])
pdf["Month"] = pdf["Date"].dt.month

pdf = pdf[(pdf["Total_Distance_Flown_NM"] > 0) &
          (pdf["Gross_Weight_At_Liftoff_kg"] > 0) &
          (pdf["Total_Fuel_Burn_kg"] > 0)].copy()

pdf["log_distance"] = np.log(pdf["Total_Distance_Flown_NM"])
pdf["log_weight"] = np.log(pdf["Gross_Weight_At_Liftoff_kg"])
pdf["log_fuel_burn"] = np.log(pdf["Total_Fuel_Burn_kg"])

print(f"Loaded {len(pdf):,} rows")
print(pdf["Fleet"].value_counts())

Loaded 1,404 rows
Fleet
B737-NG     1074
A330         283
B737F-NG      47
Name: count, dtype: int64


In [3]:
FLEET_GROUP_MAP = {
    "B737-NG": "B737",
    "B737F-NG": "B737",
    "A330": "A330",
}

pdf["Fleet_Group"] = pdf["Fleet"].map(FLEET_GROUP_MAP)

print(pdf["Fleet_Group"].value_counts())

Fleet_Group
B737    1121
A330     283
Name: count, dtype: int64


In [4]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

grp = pdf[pdf["Fleet_Group"] == "B737"]

sdf = spark.createDataFrame(
    grp[["Month", "log_distance", "log_weight",
         "Max_Tailwind_During_Takeoff_kt", "log_fuel_burn"]]
)

train = sdf.filter(sdf.Month <= 5)
test = sdf.filter(sdf.Month > 5)

print("Train rows:", train.count())
print("Test rows:", test.count())

Train rows: 831
Test rows: 290


In [5]:
assembler = VectorAssembler(
    inputCols=["log_distance", "log_weight", "Max_Tailwind_During_Takeoff_kt"],
    outputCol="features"
)

train_v = assembler.transform(train)
test_v = assembler.transform(test)

lr = LinearRegression(featuresCol="features", labelCol="log_fuel_burn")
model = lr.fit(train_v)

preds = model.transform(test_v)
preds.select("log_fuel_burn", "prediction").show(10)

+-----------------+-----------------+
|    log_fuel_burn|       prediction|
+-----------------+-----------------+
|9.055906318669118| 9.07309633066567|
|8.034631032923107|8.087584722218796|
|8.184792654165078|8.233636264112821|
|7.425357887027151|7.302248615774121|
|7.938445551164788|8.005209691594185|
|  8.9758830607617| 8.93728409559003|
|8.145549631783584|8.196643147101561|
|7.233455418621439|7.172055438971838|
|7.312553498102598|7.248013484482912|
|8.388450315523512| 8.41911944838946|
+-----------------+-----------------+
only showing top 10 rows



In [6]:
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

rmse_log = RegressionEvaluator(
    labelCol="log_fuel_burn", predictionCol="prediction", metricName="rmse"
).evaluate(preds)

mae_log = RegressionEvaluator(
    labelCol="log_fuel_burn", predictionCol="prediction", metricName="mae"
).evaluate(preds)

preds_pd = preds.select("log_fuel_burn", "prediction").toPandas()
actual_kg = np.exp(preds_pd["log_fuel_burn"])
pred_kg = np.exp(preds_pd["prediction"])

rmse_kg = np.sqrt(np.mean((actual_kg - pred_kg) ** 2))
mae_kg = np.mean(np.abs(actual_kg - pred_kg))

print(f"RMSE (log scale): {rmse_log:.4f}")
print(f"MAE  (log scale): {mae_log:.4f}")
print(f"RMSE (kg):        {rmse_kg:,.0f}")
print(f"MAE  (kg):        {mae_kg:,.0f}")

RMSE (log scale): 0.0694
MAE  (log scale): 0.0554
RMSE (kg):        519
MAE  (kg):        350


In [7]:
coefs = dict(zip(["log_distance", "log_weight", "Max_Tailwind_During_Takeoff_kt"],
                  model.coefficients.toArray()))

print(f"Intercept: {model.intercept:.4f}")
for feat, coef in coefs.items():
    print(f"{feat:35s} coef = {coef:+.4f}")

b1 = coefs["log_distance"]
print(f"\n-> a 10% longer flight burns {((1.10 ** b1) - 1) * 100:+.1f}% more fuel "
      f"(holding weight and tailwind fixed)")

Intercept: -6.4264
log_distance                        coef = +0.7166
log_weight                          coef = +0.9245
Max_Tailwind_During_Takeoff_kt      coef = -0.0031

-> a 10% longer flight burns +7.1% more fuel (holding weight and tailwind fixed)


In [8]:
grp_a330 = pdf[pdf["Fleet_Group"] == "A330"]
print(grp_a330["Month"].value_counts().sort_index())

Month
4     64
5    125
6      1
8     93
Name: count, dtype: int64


In [9]:
sdf_a330 = spark.createDataFrame(
    grp_a330[["Month", "log_distance", "log_weight",
              "Max_Tailwind_During_Takeoff_kt", "log_fuel_burn"]]
)

train_a330 = sdf_a330.filter(sdf_a330.Month <= 5)
test_a330 = sdf_a330.filter(sdf_a330.Month > 5)

print("Train rows:", train_a330.count())
print("Test rows:", test_a330.count())

Train rows: 189
Test rows: 94


In [10]:
assembler_a330 = VectorAssembler(
    inputCols=["log_distance", "log_weight", "Max_Tailwind_During_Takeoff_kt"],
    outputCol="features"
)

train_a330_v = assembler_a330.transform(train_a330)
test_a330_v = assembler_a330.transform(test_a330)

lr_a330 = LinearRegression(featuresCol="features", labelCol="log_fuel_burn")
model_a330 = lr_a330.fit(train_a330_v)

preds_a330 = model_a330.transform(test_a330_v)
preds_a330.select("log_fuel_burn", "prediction").show(10)

+------------------+------------------+
|     log_fuel_burn|        prediction|
+------------------+------------------+
|10.648966238356156|  10.6173776299838|
| 8.875566691990551| 9.096021175879283|
| 8.879472402074802|  9.05080365498296|
|10.759093660793777| 10.74245564800346|
|10.823431602908185|10.825423719073463|
| 8.286773231131251| 8.309080416487486|
|10.056337386630338|10.108492599687896|
| 8.350902451694811|  8.28868069093015|
| 10.10716282065494|10.147066278460237|
|10.754684942721584|10.742473848761698|
+------------------+------------------+
only showing top 10 rows



In [11]:
rmse_log_a330 = RegressionEvaluator(
    labelCol="log_fuel_burn", predictionCol="prediction", metricName="rmse"
).evaluate(preds_a330)

mae_log_a330 = RegressionEvaluator(
    labelCol="log_fuel_burn", predictionCol="prediction", metricName="mae"
).evaluate(preds_a330)

preds_a330_pd = preds_a330.select("log_fuel_burn", "prediction").toPandas()
actual_kg_a330 = np.exp(preds_a330_pd["log_fuel_burn"])
pred_kg_a330 = np.exp(preds_a330_pd["prediction"])

rmse_kg_a330 = np.sqrt(np.mean((actual_kg_a330 - pred_kg_a330) ** 2))
mae_kg_a330 = np.mean(np.abs(actual_kg_a330 - pred_kg_a330))

print(f"RMSE (log scale): {rmse_log_a330:.4f}")
print(f"MAE  (log scale): {mae_log_a330:.4f}")
print(f"RMSE (kg):        {rmse_kg_a330:,.0f}")
print(f"MAE  (kg):        {mae_kg_a330:,.0f}")

RMSE (log scale): 0.0583
MAE  (log scale): 0.0378
RMSE (kg):        1,014
MAE  (kg):        791


In [12]:
coefs_a330 = dict(zip(["log_distance", "log_weight", "Max_Tailwind_During_Takeoff_kt"],
                        model_a330.coefficients.toArray()))

print(f"Intercept: {model_a330.intercept:.4f}")
for feat, coef in coefs_a330.items():
    print(f"{feat:35s} coef = {coef:+.4f}")

b1_a330 = coefs_a330["log_distance"]
print(f"\n-> a 10% longer flight burns {((1.10 ** b1_a330) - 1) * 100:+.1f}% more fuel "
      f"(holding weight and tailwind fixed)")

Intercept: -8.9322
log_distance                        coef = +0.6803
log_weight                          coef = +1.1476
Max_Tailwind_During_Takeoff_kt      coef = +0.0015

-> a 10% longer flight burns +6.7% more fuel (holding weight and tailwind fixed)


In [13]:
import pickle

fitted_models = {
    "B737": {
        "intercept": float(model.intercept),
        "coefficients": dict(zip(
            ["log_distance", "log_weight", "Max_Tailwind_During_Takeoff_kt"],
            model.coefficients.toArray().tolist()
        )),
    },
    "A330": {
        "intercept": float(model_a330.intercept),
        "coefficients": dict(zip(
            ["log_distance", "log_weight", "Max_Tailwind_During_Takeoff_kt"],
            model_a330.coefficients.toArray().tolist()
        )),
    },
}

with open("spark_fuel_models.pkl", "wb") as f:
    pickle.dump(fitted_models, f)

print("Saved:", fitted_models)

Saved: {'B737': {'intercept': -6.426394063949497, 'coefficients': {'log_distance': 0.7166201686824134, 'log_weight': 0.9244778178162769, 'Max_Tailwind_During_Takeoff_kt': -0.003064840113961984}}, 'A330': {'intercept': -8.93222388124341, 'coefficients': {'log_distance': 0.6802848191599308, 'log_weight': 1.1475673176091954, 'Max_Tailwind_During_Takeoff_kt': 0.0015051430171990363}}}


In [14]:
pdf["Fleet_Group"] = pdf["Fleet"].map(FLEET_GROUP_MAP)
print(pdf["Fleet_Group"].value_counts())

Fleet_Group
B737    1121
A330     283
Name: count, dtype: int64


In [15]:
import dill

with open("kde_models_3d.pkl", "rb") as f:
    kde_models_3d = dill.load(f)

def generate_flight_inputs(fleet, max_attempts=50, rng=None):
    f = kde_models_3d[fleet]
    kde = f["kde"]
    rng = rng or np.random.default_rng()

    for _ in range(max_attempts):
        sample = kde.resample(1, seed=rng)
        distance = float(np.exp(sample[0, 0]))
        weight = float(np.exp(sample[1, 0]))
        tailwind = float(sample[2, 0])

        if (f["dist_min"] <= distance <= f["dist_max"] and
                f["weight_min"] <= weight <= f["weight_max"] and
                f["tailwind_min"] <= tailwind <= f["tailwind_max"]):
            return distance, weight, tailwind

    distance = min(max(distance, f["dist_min"]), f["dist_max"])
    weight = min(max(weight, f["weight_min"]), f["weight_max"])
    tailwind = min(max(tailwind, f["tailwind_min"]), f["tailwind_max"])
    return distance, weight, tailwind

print(list(kde_models_3d.keys()))

['A330', 'B737-NG', 'B737F-NG']


In [16]:
import pickle

with open("spark_fuel_models.pkl", "rb") as f:
    fitted_models = pickle.load(f)

print(fitted_models)

{'B737': {'intercept': -6.426394063949497, 'coefficients': {'log_distance': 0.7166201686824134, 'log_weight': 0.9244778178162769, 'Max_Tailwind_During_Takeoff_kt': -0.003064840113961984}}, 'A330': {'intercept': -8.93222388124341, 'coefficients': {'log_distance': 0.6802848191599308, 'log_weight': 1.1475673176091954, 'Max_Tailwind_During_Takeoff_kt': 0.0015051430171990363}}}


In [17]:
def predict_fuel_burn(fleet_group, distance, weight, tailwind):
    m = fitted_models[fleet_group]
    log_pred = (
        m["intercept"]
        + m["coefficients"]["log_distance"] * np.log(distance)
        + m["coefficients"]["log_weight"] * np.log(weight)
        + m["coefficients"]["Max_Tailwind_During_Takeoff_kt"] * tailwind
    )
    return float(np.exp(log_pred))


rng = np.random.default_rng()

for fleet in kde_models_3d:
    distance, weight, tailwind = generate_flight_inputs(fleet, rng=rng)

    fleet_group = "B737" if "B737" in fleet else "A330"
    predicted_fuel = predict_fuel_burn(fleet_group, distance, weight, tailwind)

    print(f"{fleet}:")
    print(f"  distance={distance:.0f} NM  weight={weight:.0f} kg  tailwind={tailwind:.1f} kt")
    print(f"  predicted fuel burn: {predicted_fuel:,.0f} kg\n")

A330:
  distance=197 NM  weight=151582 kg  tailwind=-3.1 kt
  predicted fuel burn: 4,209 kg

B737-NG:
  distance=1294 NM  weight=68618 kg  tailwind=-0.1 kt
  predicted fuel burn: 8,139 kg

B737F-NG:
  distance=557 NM  weight=53136 kg  tailwind=-5.3 kt
  predicted fuel burn: 3,569 kg

